# TEI HTML Parser for Italian Music Theory Treatises

## Overview
This notebook processes TMI (Thesaurus Musicarum Italicarum) TEI HTML files of Italian music theory treatises and creates a vector database for semantic search using LangChain and ChromaDB.

## Key Features of This Notebook
This notebook now uses an intelligent incremental update approach:

1. CONFIGURATION TRACKING (db_config.json)
   - Tracks embedding model, chunk size, and other settings
   - Only recreates DB when breaking changes are detected
   - Breaking changes: embedding model or chunk size changes

2. FILE CHANGE DETECTION (file_hashes.json)
   - Tracks MD5 hash of each source HTML file
   - Only reprocesses files that have changed
   - Skips unchanged files automatically

3. DOCUMENT ID MANAGEMENT
   - Each chunk gets a unique, deterministic ID
   - Allows updating existing documents without duplicates
   - Format: MD5(source_file_page_number_chunk_index)

4. INCREMENTAL UPDATES
   - When a file changes, old documents are deleted first
   - New documents are added with same IDs if content unchanged
   - Prevents duplicate embeddings

### Common Operations:

```python
# Add or update files
process_html_files()  # Only processes changed files

# Force complete rebuild (rare)
process_html_files(force_reprocess=True)

# View database statistics
get_db_stats()

# Remove a specific file
delete_source_file("artart.html")

# Search the database
results = vector_store.similarity_search("your query here", k=5)

# Search with scores
results = vector_store.similarity_search_with_score("your query", k=5)
```


## Main Steps:
1. **Import Libraries** - Load necessary Python packages
2. **Configure Database** - Set up ChromaDB with intelligent update tracking
3. **Parse TEI HTML** - Extract metadata and text from treatises
4. **Create Embeddings** - Generate vector embeddings using OpenAI
5. **Store & Query** - Save to ChromaDB and explore the database

---

## Step 1: Import Required Libraries and API Key

In [1]:
# Standard library imports
import glob
import hashlib
import json
import os
import re
import shutil
import getpass
from pathlib import Path

# Third-party imports
import pandas as pd
from bs4 import BeautifulSoup, NavigableString

# LangChain imports
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/rfreedma/anaconda3/envs/lang/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## Step 2: Configure OpenAI API Key and ChromaDB Settings

We use OpenAI's embedding model to convert text into vectors for semantic search.

In [2]:
# Prompt for OpenAI API key with password masking
print("Please enter your OpenAI API key:")
openai_api_key = getpass.getpass("API Key: ")

# Set as environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key

# Verify it was set (show only first/last few characters for security)
if openai_api_key:
    masked_key = f"{openai_api_key[:7]}...{openai_api_key[-4:]}"
    print(f"✓ API key set successfully: {masked_key}")
else:
    print("✗ No API key entered")

Please enter your OpenAI API key:
✓ API key set successfully: sk-proj...jXgA


In [3]:
# Configuration for database schema and settings--these will be passed to all the relevant components below
DB_CONFIG = {
    "version": "1.0",
    "embedding_model": "text-embedding-3-large",
    "chunk_size": 2000,
    "chunk_overlap": 300,
    "collection_name": "HTML_samples_italian"
}

db_path = Path('./chroma-db_italian')
config_path = db_path / 'db_config.json'

# Check if we need to recreate the database
should_recreate = False

if db_path.exists() and config_path.exists():
    # Load existing config
    with open(config_path, 'r') as f:
        existing_config = json.load(f)
    
    # Check for breaking changes
    if (existing_config.get('embedding_model') != DB_CONFIG['embedding_model'] or
        existing_config.get('chunk_size') != DB_CONFIG['chunk_size']):
        print(f"⚠️  Breaking changes detected:")
        print(f"   Old: {existing_config}")
        print(f"   New: {DB_CONFIG}")
        should_recreate = True
    else:
        print(f"✓ Using existing database - configuration unchanged")
        print(f"  Will perform incremental updates only")
elif not db_path.exists():
    print(f"✓ Creating new database at {db_path}")
    should_recreate = True
else:
    print(f"⚠️  Database exists but no config found - will recreate")
    should_recreate = True

# Delete database only if necessary
if should_recreate and db_path.exists():
    shutil.rmtree(db_path)
    print(f"✓ Deleted existing database at {db_path}")

# Create directory if needed
db_path.mkdir(exist_ok=True)

# Save current configuration
with open(config_path, 'w') as f:
    json.dump(DB_CONFIG, f, indent=2)

# Initialize embeddings
embeddings = OpenAIEmbeddings(model=DB_CONFIG['embedding_model'])

# Initialize Chroma vector store
vector_store = Chroma(
    collection_name=DB_CONFIG['collection_name'],
    embedding_function=embeddings,
    persist_directory=str(db_path)
)

# Configure text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=DB_CONFIG['chunk_size'],
    chunk_overlap=DB_CONFIG['chunk_overlap'],
    length_function=len,
    is_separator_regex=False
)

✓ Using existing database - configuration unchanged
  Will perform incremental updates only


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


---

## Step 3: Initialize ChromaDB Vector Store

### What is a Vector Database?
A vector database stores text as numerical vectors (embeddings) that capture semantic meaning. This allows us to find relevant passages based on meaning, not just keyword matching.

### Intelligent Update Strategy
This notebook implements a **smart incremental update system**:
- **Configuration Tracking**: Detects breaking changes (embedding model, chunk size)
- **File Change Detection**: Only reprocesses modified HTML files
- **No Unnecessary Rebuilds**: Saves time and API costs

The system creates two tracking files:
- `db_config.json` - Stores database configuration
- `file_hashes.json` - Tracks which files have been processed

In [4]:
# Metadata extraction function for TMI Italian music theory files
# The metadata will be used in the chroma db and also in the csv file for the Streamlit app as a record of our sources

def extract_metadata(soup, html_path):
    """
    Extract metadata from TMI HTML file.
    
    Metadata is in <ul class="TitleInfo"> with:
    - <span class="bannerAuthor"> for author
    - <span class="bannerTitle"> for title
    - <span class="bannerDates"> for date
    """
    # Convert to Path object if it's a string
    if isinstance(html_path, str):
        html_path = Path(html_path)
    
    # The TitleInfo is in the nav element, not in main
    metadata_div = soup.find("ul", class_="TitleInfo")
    if not metadata_div:
        return None

    metadata_item = metadata_div.find("li")
    if not metadata_item:
        return None

    # Add error handling for each span element
    title_span = metadata_item.find("span", class_="bannerTitle")
    author_span = metadata_item.find("span", class_="bannerAuthor")
    date_span = metadata_item.find("span", class_="bannerDates")
    
    title = title_span.get_text(strip=True) if title_span else "Unknown Title"
    author = author_span.get_text(strip=True) if author_span else "Unknown Author"
    # Remove parentheses and strip whitespace from date
    if date_span:
        date = date_span.get_text(strip=True).replace("(", "").replace(")", "").strip()
    else:
        date = "Unknown Date"

    return {
        "Filename": html_path.name,
        "Title": title,
        "Author": author,
        "Date": date,
        "Source": f"https://tmiweb.science.uu.nl/text/reading-edition/{html_path.name}"
    }

# Main page content extraction
def extract_pages_and_text(html_content, html_path):
    """
    Extract page numbers and their associated text from TMI TEI HTML.
    
    Page numbers are in <span class="tei pb"><span class="tei pbSpan">page X</span></span>
    Page content follows each page break as sibling elements until the next page break.
    
    Args:
        html_content: Either a string containing TEI HTML or a BeautifulSoup object
        html_path: Path to the HTML file    
        
    Returns:
        List of dictionaries with 'PageNumber', 'PageText', and metadata
    """
    # Check if html_content is already a BeautifulSoup object
    if isinstance(html_content, BeautifulSoup):
        soup = html_content
    else:
        soup = BeautifulSoup(html_content, 'html.parser')

    metadata = extract_metadata(soup, html_path)
    
    # Find all page break tags - these are <span class="tei pb"> elements
    page_breaks = soup.find_all('span', class_='tei pb')
    
    results = []
    
    for i, pb in enumerate(page_breaks):
        # Extract page number from the nested pbSpan
        # Structure: <span class="tei pb"><span class="tei pbSpan">page X</span></span>
        page_span = pb.find('span', class_='tei pbSpan')
        page_number = page_span.get_text(strip=True) if page_span else f"page {i+1}"
        
        # Collect ALL text content that follows this page break until the next one
        # Strategy: Get all text nodes between the two page breaks in document order
        
        # Get the next page break to know where to stop
        next_pb = page_breaks[i + 1] if i + 1 < len(page_breaks) else None
        
        # Collect all NavigableStrings (text nodes) between page breaks
        page_text_parts = []
        
        # Start after current page break
        for element in pb.next_elements:
            # Stop if we hit the next page break
            if next_pb and element == next_pb:
                break
            
            # Only collect NavigableString objects (actual text nodes)
            # This avoids double-counting when we have nested tags
            # Note that we are NOT yet collecting information about images and figures, just text
            if isinstance(element, NavigableString):
                text = str(element).strip()
                if text:
                    page_text_parts.append(text)
        
        page_text = ' '.join(page_text_parts)
        # This is the assembled text content for this page number
        result_entry = {
            'PageNumber': page_number,
            'PageText': page_text
        }
        
        # Attach metadata if available
        if metadata:
            result_entry.update(metadata)
        
        results.append(result_entry)
    
    return results

---

## Step 4: Define TEI HTML Parsing Functions

These functions extract structured data from TMI TEI HTML files:

`extract_metadata(soup, html_path)`
Extracts bibliographic information from the navigation header:
- Title of the treatise
- Author name
- Date of publication
- Source URL

`extract_pages_and_text(html_content, html_path)`
Extracts page-by-page text content:
- Identifies page breaks (`<span class="tei pb">` tags)
- Collects all text between page breaks
- Preserves document structure
- Returns list of pages with metadata

---

## Step 5: Test Text Extraction (Optional)

**Purpose**: This cell is for testing/debugging only. It extracts text and metadata from HTML files without creating embeddings.


**Note**: You can skip this cell and go directly to `process_html_files()` for normal operation.

In [5]:
# Run this on all files in the source directory: this is just to get the text, not the vector db
tmi_dir = Path("italian_sources")
for html_path in tmi_dir.glob("*.html"):
    with html_path.open("r", encoding="utf-8") as handle:
        html_content = handle.read()    
        results = extract_pages_and_text(html_content, html_path)

In [6]:
# Just to check one result
results[0]

{'PageNumber': 'page i',
 'PageText': "page i DIALOGO DEL R. M. DON PIETRO PONTIO PARMIGIANO, Oue si tratta della Theorica, e Prattica di Musica. \n                        Et anco si mostra la diuersità de' Contraponti, & Canoni. [Figure] In Parma , appresso Erasmo Viothi . 1595. Con licenza de' Superiori.",
 'Filename': 'pondia.html',
 'Title': 'Dialogo di musica',
 'Author': 'Pietro Pontio',
 'Date': '1595',
 'Source': 'https://tmiweb.science.uu.nl/text/reading-edition/pondia.html'}

---
## Get the Metadata DataFrame

In [7]:
# Test metadata df
tmi_dir = Path("italian_sources")
metadata = []
for html_path in tmi_dir.glob("*.html"):
    with html_path.open("r", encoding="utf-8") as handle:
        html_content = handle.read()    
        one_document_metadata = extract_metadata(BeautifulSoup(html_content, 'html.parser'), html_path)  
        metadata.append(one_document_metadata)   
metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv("italian_tmi_metadata.csv", index=False)
metadata_df

,Filename,Title,Author,Date,Source
0,artart.html,L'Artusi,Giovanni Maria Artusi,1600,https://tmiweb.science.uu.nl/text/reading-edit...
1,zarins58.html,Le istitutioni harmoniche,Gioseffo Zarlino,1558,https://tmiweb.science.uu.nl/text/reading-edit...
2,tigcom.html,Il compendio della musica,Orazio Tigrini,1588,https://tmiweb.science.uu.nl/text/reading-edit...
3,vicant.html,L'antica musica ridotta alla moderna prattica,Nicola Vicentino,1555,https://tmiweb.science.uu.nl/text/reading-edit...
4,pondia.html,Dialogo di musica,Pietro Pontio,1595,https://tmiweb.science.uu.nl/text/reading-edit...


## Functions to Process all the Files and Create/Update the ChromaDB

In [8]:
def generate_document_id(source_file, page_number, chunk_index):
    """Generate a unique, deterministic ID for each document chunk."""
    id_string = f"{source_file}_{page_number}_chunk_{chunk_index}"
    return hashlib.md5(id_string.encode()).hexdigest()

def get_file_hash(filepath):
    """Get MD5 hash of a file to detect changes."""
    hash_md5 = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def load_file_hashes():
    """Load previously processed file hashes."""
    hash_file = db_path / 'file_hashes.json'
    if hash_file.exists():
        with open(hash_file, 'r') as f:
            return json.load(f)
    return {}

def save_file_hashes(hashes):
    """Save file hashes to track what's been processed."""
    hash_file = db_path / 'file_hashes.json'
    with open(hash_file, 'w') as f:
        json.dump(hashes, f, indent=2)

def process_html_files(html_dir='italian_sources', force_reprocess=False, batch_size=100):
    """
    Process all TMI TEI HTML files with BATCH PROCESSING to avoid OpenAI token limits.
    
    Args:
        html_dir: Directory containing HTML files
        force_reprocess: If True, reprocess all files regardless of changes
        batch_size: Number of chunks to process at once (default 100)
    """
    html_files = glob.glob(os.path.join(html_dir, '*.html'))
    
    if not html_files:
        print(f"No TMI HTML files found in {html_dir}")
        return
    
    # Load existing file hashes to detect changes
    existing_hashes = load_file_hashes()
    new_hashes = {}
    
    total_chunks = 0
    total_pages = 0
    files_processed = 0
    files_skipped = 0
    files_updated = 0
    
    for filepath in html_files:
        try:
            filename = os.path.basename(filepath)
            current_hash = get_file_hash(filepath)
            new_hashes[filename] = current_hash
            
            # Skip if file hasn't changed (unless force_reprocess is True)
            if not force_reprocess and filename in existing_hashes:
                if existing_hashes[filename] == current_hash:
                    print(f"⊙ {filename} - No changes, skipping")
                    files_skipped += 1
                    continue
                else:
                    print(f"↻ {filename} - File changed, updating...")
                    files_updated += 1
                    # Delete old documents for this file
                    source_id = f"https://tmiweb.science.uu.nl/text/reading-edition/{filename}"
                    try:
                        vector_store.delete(where={"source": source_id})
                        print(f"  Deleted old documents for {filename}")
                    except Exception as e:
                        print(f"  Note: Could not delete old documents: {e}")
            else:
                print(f"+ {filename} - New file, processing...")
            
            with open(filepath, 'r', encoding='utf-8') as f:
                html_content = f.read()
            
            # Parse HTML to return the pages in the source document, with metadata
            pages = extract_pages_and_text(html_content, Path(filepath))
            
            if not pages:
                print(f"  Warning: No pages found in {filepath}")
                continue
            
            file_chunks = 0
            chunk_counter = 0
            all_chunk_ids = []
            all_chunks = []
            
            # Chunk each page in the source document separately
            for page in pages:
                # Create metadata for this page
                page_metadata = {
                    "title": page['Title'],
                    "author": page['Author'],
                    "date": page['Date'],
                    "source": page['Source'],
                    "page_number": page['PageNumber'],
                    "filename": filename
                }
                
                # Split page text into chunks
                chunks = text_splitter.create_documents(
                    texts=[page['PageText']],
                    metadatas=[page_metadata]
                )
                
                # Generate IDs for each chunk
                for chunk in chunks:
                    chunk_id = generate_document_id(
                        page['Source'], 
                        page['PageNumber'], 
                        chunk_counter
                    )
                    all_chunk_ids.append(chunk_id)
                    all_chunks.append(chunk)
                    chunk_counter += 1
                
                file_chunks += len(chunks)
            
            # *** BATCH PROCESSING TO AVOID OPENAI TOKEN LIMITS ***
            if all_chunks:
                num_batches = (len(all_chunks) + batch_size - 1) // batch_size
                print(f"  Processing {len(all_chunks)} chunks in {num_batches} batch(es)...")
                
                for i in range(0, len(all_chunks), batch_size):
                    batch_chunks = all_chunks[i:i + batch_size]
                    batch_ids = all_chunk_ids[i:i + batch_size]
                    
                    # Add batch to vector store
                    vector_store.add_documents(
                        documents=batch_chunks,
                        ids=batch_ids
                    )
                    
                    if num_batches > 1:
                        batch_num = (i // batch_size) + 1
                        print(f"    Batch {batch_num}/{num_batches} complete ({len(batch_chunks)} chunks)")
            
            total_chunks += file_chunks
            total_pages += len(pages)
            files_processed += 1
            
            print(f'  ✓ Title: {page["Title"][:80]}...' if len(page["Title"]) > 80 else f'  ✓ Title: {page["Title"]}')
            print(f'    Author: {page["Author"]} | Date: {page["Date"]}')
            print(f'    Pages: {len(pages)} | Chunks: {file_chunks}')
            
        except Exception as e:
            print(f"✗ Error processing {filepath}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # Save the new hashes
    save_file_hashes(new_hashes)
    
    print(f'\n{"="*50}')
    print(f'ChromaDB Processing Complete')
    print(f'{"="*50}')
    print(f'Total files: {len(html_files)}')
    print(f'  New/Updated: {files_processed}')
    print(f'  Skipped (unchanged): {files_skipped}')
    print(f'  Changed: {files_updated}')
    print(f'Total pages processed: {total_pages}')
    print(f'Total chunks added: {total_chunks}')

print("✓ Functions updated - batch processing enabled")

✓ Functions updated - batch processing enabled


---

## Step 6: Process HTML Files and Create Vector Database

### What Happens Here:

1. **File Hash Tracking**: Computes MD5 hash of each HTML file to detect changes
2. **Smart Processing**: 
   - ⊙ **Skips** unchanged files
   - ↻ **Updates** modified files (deletes old, adds new)
   - + **Processes** new files
3. **Text Chunking**: Splits long pages into 2000-character chunks with 300-char overlap
4. **ID Generation**: Creates deterministic IDs for each chunk (prevents duplicates)
5. **Embedding Creation**: Sends chunks to OpenAI for vector embedding
6. **Database Storage**: Stores embeddings and metadata in ChromaDB

### Understanding Chunks vs Pages:
- **Page**: A logical division from the original document (marked by `<span class="tei pb">` tags)
- **Chunk**: A piece of text ≤2000 characters for optimal embedding
- One page may create multiple chunks if text is long

In [9]:
# Process HTML files - only processes new or changed files by default
# Use force_reprocess=True to reprocess everything
process_html_files(html_dir='italian_sources', force_reprocess=True)

+ artart.html - New file, processing...
  Processing 314 chunks in 4 batch(es)...
    Batch 1/4 complete (100 chunks)
    Batch 2/4 complete (100 chunks)
    Batch 3/4 complete (100 chunks)
    Batch 4/4 complete (14 chunks)
  ✓ Title: L'Artusi
    Author: Giovanni Maria Artusi | Date: 1600
    Pages: 155 | Chunks: 314
+ zarins58.html - New file, processing...
  Processing 1027 chunks in 11 batch(es)...
    Batch 1/11 complete (100 chunks)
    Batch 2/11 complete (100 chunks)
    Batch 3/11 complete (100 chunks)
    Batch 4/11 complete (100 chunks)
    Batch 5/11 complete (100 chunks)
    Batch 6/11 complete (100 chunks)
    Batch 7/11 complete (100 chunks)
    Batch 8/11 complete (100 chunks)
    Batch 9/11 complete (100 chunks)
    Batch 10/11 complete (100 chunks)
    Batch 11/11 complete (27 chunks)
  ✓ Title: Le istitutioni harmoniche
    Author: Gioseffo Zarlino | Date: 1558
    Pages: 353 | Chunks: 1027
+ tigcom.html - New file, processing...
  Processing 254 chunks in 3 batch(e

In [10]:
# For checking page and character counts
pages = results
# Calculate lengths of all PageText entries
page_lengths = [len(page['PageText']) for page in pages]
avg_length = sum(page_lengths) / len(page_lengths)
    
print(f"  Total pages: {len(pages)}")
print(f"  Average PageText length: {avg_length:.2f} characters")
print(f"  Min length: {min(page_lengths)} characters")
print(f"  Max length: {max(page_lengths)} characters")

  Total pages: 160
  Average PageText length: 1748.12 characters
  Min length: 7 characters
  Max length: 3424 characters


In [11]:
# Helper functions for database management

def get_db_stats():
    """Get statistics about the current database."""
    all_docs = vector_store.get()
    
    if not all_docs or 'metadatas' not in all_docs:
        print("Database is empty")
        return
    
    total_docs = len(all_docs['ids'])
    
    # Collect statistics
    authors = set()
    sources = set()
    dates = set()
    
    for metadata in all_docs['metadatas']:
        if metadata:
            if 'author' in metadata:
                authors.add(metadata['author'])
            if 'source' in metadata:
                sources.add(metadata['source'])
            if 'date' in metadata:
                dates.add(metadata['date'])
    
    print(f"Database Statistics:")
    print(f"  Total chunks: {total_docs}")
    print(f"  Unique authors: {len(authors)}")
    print(f"  Unique sources: {len(sources)}")
    print(f"  Date range: {sorted(dates)}")
    print(f"\nAuthors: {sorted(authors)}")
    
    return {
        'total_docs': total_docs,
        'authors': sorted(authors),
        'sources': sorted(sources),
        'dates': sorted(dates)
    }

def delete_source_file(source_filename):
    """Delete all documents from a specific source file."""
    source_id = f"https://tmiweb.science.uu.nl/text/reading-edition/{source_filename}"
    try:
        vector_store.delete(where={"source": source_id})
        print(f"✓ Deleted all documents from {source_filename}")
        
        # Also remove from file hashes
        hashes = load_file_hashes()
        if source_filename in hashes:
            del hashes[source_filename]
            save_file_hashes(hashes)
            print(f"✓ Removed {source_filename} from tracking")
    except Exception as e:
        print(f"✗ Error deleting documents: {e}")

# Get current database statistics
get_db_stats()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Database Statistics:
  Total chunks: 2393
  Unique authors: 5
  Unique sources: 5
  Date range: ['1555', '1558', '1588', '1595', '1600']

Authors: ['Gioseffo Zarlino', 'Giovanni Maria Artusi', 'Nicola Vicentino', 'Orazio Tigrini', 'Pietro Pontio']


{'total_docs': 2393,
 'authors': ['Gioseffo Zarlino',
  'Giovanni Maria Artusi',
  'Nicola Vicentino',
  'Orazio Tigrini',
  'Pietro Pontio'],
 'sources': ['https://tmiweb.science.uu.nl/text/reading-edition/artart.html',
  'https://tmiweb.science.uu.nl/text/reading-edition/pondia.html',
  'https://tmiweb.science.uu.nl/text/reading-edition/tigcom.html',
  'https://tmiweb.science.uu.nl/text/reading-edition/vicant.html',
  'https://tmiweb.science.uu.nl/text/reading-edition/zarins58.html'],
 'dates': ['1555', '1558', '1588', '1595', '1600']}

---

## Step 7: Explore Database Contents

These cells demonstrate what's stored in the database and how to access it.

### 7.1: View Sample Metadata

Each chunk in the database has metadata that describes its source.

In [12]:
# Get a sample of documents from the database
sample_docs = vector_store.get(limit=3)

# Display metadata from first 3 chunks
print("=" * 60)
print("SAMPLE METADATA FROM DATABASE")
print("=" * 60)

for i, metadata in enumerate(sample_docs['metadatas'][:3], 1):
    print(f"\n📄 Chunk {i}:")
    print(f"   Title: {metadata.get('title', 'N/A')}")
    print(f"   Author: {metadata.get('author', 'N/A')}")
    print(f"   Date: {metadata.get('date', 'N/A')}")
    print(f"   Page Number: {metadata.get('page_number', 'N/A')}")
    print(f"   Source File: {metadata.get('filename', 'N/A')}")
    print(f"   Source URL: {metadata.get('source', 'N/A')}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


SAMPLE METADATA FROM DATABASE

📄 Chunk 1:
   Title: L'Artusi
   Author: Giovanni Maria Artusi
   Date: 1600
   Page Number: page i
   Source File: artart.html
   Source URL: https://tmiweb.science.uu.nl/text/reading-edition/artart.html

📄 Chunk 2:
   Title: L'Artusi
   Author: Giovanni Maria Artusi
   Date: 1600
   Page Number: page ii
   Source File: artart.html
   Source URL: https://tmiweb.science.uu.nl/text/reading-edition/artart.html

📄 Chunk 3:
   Title: L'Artusi
   Author: Giovanni Maria Artusi
   Date: 1600
   Page Number: page iii
   Source File: artart.html
   Source URL: https://tmiweb.science.uu.nl/text/reading-edition/artart.html


### 7.2: View Sample Text Chunks

See what the actual text chunks look like.

In [13]:
# Display text content from sample chunks
print("=" * 60)
print("SAMPLE TEXT CHUNKS")
print("=" * 60)

for i, (doc_text, metadata) in enumerate(zip(sample_docs['documents'][:3], sample_docs['metadatas'][:3]), 1):
    print(f"\n📝 Chunk {i}:")
    print(f"   Source: {metadata.get('title', 'N/A')} (Page {metadata.get('page_number', 'N/A')})")
    print(f"   Length: {len(doc_text)} characters")
    print(f"   Text Preview:")
    print(f"   {'-' * 55}")
    # Show first 300 characters
    preview = doc_text[:300] + "..." if len(doc_text) > 300 else doc_text
    print(f"   {preview}")
    print()

SAMPLE TEXT CHUNKS

📝 Chunk 1:
   Source: L'Artusi (Page page i)
   Length: 579 characters
   Text Preview:
   -------------------------------------------------------
   page i L'ARTVSI 
                        Ouero 
                        DELLE IMPERFETTIONI 
                        DELLA MODERNA MVSICA 
                        Ragionamenti dui. 
                        Ne' quali si ragiona di molte cose vtili, & neessarie alli 
                        Moderni Co...


📝 Chunk 2:
   Source: L'Artusi (Page page ii)
   Length: 1535 characters
   Text Preview:
   -------------------------------------------------------
   page ii ALL'ILLVSTRISS. 
                              ET REVERENDISS. 
                              SIGNORE 
                              IL CARDINALE POMPEO 
                              ARIGONI. ECco finalmente Illustriss. & Reuerendis simo Signore; che io mando al publico, le 
               ...


📝 Chunk 3:
   Source: L'Artusi (Page page iii)
   Length: 1633 cha

### 7.3: View Sample Vectors (Embeddings)

**What are vectors?** Numerical representations of text meaning. OpenAI's `text-embedding-3-large` creates 3072-dimensional vectors.

**Why vectors?** They enable semantic search - finding text with similar *meaning*, not just matching keywords.

In [14]:
# Display information about the embedding vectors
import numpy as np

if 'embeddings' in sample_docs and sample_docs['embeddings']:
    print("=" * 60)
    print("SAMPLE EMBEDDING VECTORS")
    print("=" * 60)
    
    for i, (embedding, metadata) in enumerate(zip(sample_docs['embeddings'][:3], sample_docs['metadatas'][:3]), 1):
        print(f"\n🔢 Chunk {i} Vector:")
        print(f"   Source: {metadata.get('title', 'N/A')} (Page {metadata.get('page_number', 'N/A')})")
        print(f"   Vector Dimensions: {len(embedding)}")
        print(f"   Vector Type: {type(embedding)}")
        print(f"   First 10 values: {embedding[:10]}")
        print(f"   Vector stats:")
        print(f"      - Min value: {min(embedding):.6f}")
        print(f"      - Max value: {max(embedding):.6f}")
        print(f"      - Mean value: {np.mean(embedding):.6f}")
        print(f"      - Std deviation: {np.std(embedding):.6f}")
else:
    print("Note: Embeddings not included in sample. Use include=['embeddings'] when calling get().")
    print("\nTo retrieve with embeddings:")
    print("sample_with_embeddings = vector_store.get(limit=3, include=['embeddings', 'documents', 'metadatas'])")

Note: Embeddings not included in sample. Use include=['embeddings'] when calling get().

To retrieve with embeddings:
sample_with_embeddings = vector_store.get(limit=3, include=['embeddings', 'documents', 'metadatas'])


## One Sample Document from the Vector Store

In [15]:
sample_with_embeddings = vector_store.get(limit=1, include=['embeddings', 'documents', 'metadatas'])

# How many dimensions in the embedding vector?
len(sample_with_embeddings['embeddings'][0])

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


3072

### 7.4: Perform a Semantic Search

Demonstrate how to search the database by meaning.

In [16]:
# Example: Search for passages about musical consonance
query = "What is consonance?"

print("=" * 60)
print(f"SEMANTIC SEARCH EXAMPLE")
print("=" * 60)
print(f"\nQuery: '{query}'")
print("\nTop 3 most relevant passages:\n")

# Perform similarity search
results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"{'='*60}")
    print(f"Result {i}:")
    print(f"   Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Author: {doc.metadata.get('author', 'N/A')}")
    print(f"   Page: {doc.metadata.get('page_number', 'N/A')}")
    print(f"   Date: {doc.metadata.get('date', 'N/A')}")
    print(f"\n   Text excerpt:")
    print(f"   {'-'*55}")
    # Show first 400 characters
    preview = doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content
    print(f"   {preview}")
    print()

SEMANTIC SEARCH EXAMPLE

Query: 'What is consonance?'

Top 3 most relevant passages:



Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Result 1:
   Title: Il compendio della musica
   Author: Orazio Tigrini
   Page: page 3
   Date: 1588

   Text excerpt:
   -------------------------------------------------------
   vna percussione d'Aria indissoluta in sino all'vdito, che può nascere da i 
                              corpi duri, & inanimati; & la voce, è vna percussione d'aria respirata, la 
                              quale nasce solo da i corpi animati; onde il Filosofo. Vox autem sonus est 
                                 quidam animati. Di maniera che ogni voce è suono, non già per lo contra rio, og...

Result 2:
   Title: Le istitutioni harmoniche
   Author: Gioseffo Zarlino
   Page: page 80
   Date: 1558

   Text excerpt:
   -------------------------------------------------------
   page 80 però essendo la Disson an za contraria alla Consonanza, non sarà diffcile saper quello, che ella sia: Imperoche è 
                              mistura di suono graue, & di acuto, la quale aspramente peruiene alle nostr

### 7.5: Search with Similarity Scores

See the actual similarity scores to understand how close the matches are.

In [17]:
# Search with similarity scores (lower distance = more similar)
query = "tuning systems and temperament"

print("=" * 60)
print(f"SEMANTIC SEARCH WITH SCORES")
print("=" * 60)
print(f"\nQuery: '{query}'")
print("\nResults ranked by similarity:\n")

# Perform similarity search with scores
results_with_scores = vector_store.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"{'='*60}")
    print(f"Result {i} - Similarity Score: {score:.4f}")
    print(f"   Title: {doc.metadata.get('title', 'N/A')}")
    print(f"   Author: {doc.metadata.get('author', 'N/A')}")
    print(f"   Page: {doc.metadata.get('page_number', 'N/A')}")
    print(f"\n   Text excerpt (first 250 chars):")
    print(f"   {'-'*55}")
    preview = doc.page_content[:250] + "..." if len(doc.page_content) > 250 else doc.page_content
    print(f"   {preview}")
    print()

print("\n💡 Note: Lower scores indicate higher similarity (distance metric)")
print("   Typical range: 0.0 (identical) to 2.0 (very different)")

SEMANTIC SEARCH WITH SCORES

Query: 'tuning systems and temperament'

Results ranked by similarity:

Result 1 - Similarity Score: 0.9516
   Title: Le istitutioni harmoniche
   Author: Gioseffo Zarlino
   Page: page 126

   Text excerpt (first 250 chars):
   -------------------------------------------------------
   page 126 cesse a tal temperamento, sotto le proportioni, o forme, le quali hora vsiamo: non però sotto alcuna di quelle, 
                              che di sopra in molte diuisioni hò mostrato: percioche sarebbe stato impossibile, di osseruare il ...

Result 2 - Similarity Score: 0.9914
   Title: Le istitutioni harmoniche
   Author: Gioseffo Zarlino
   Page: page 125

   Text excerpt (first 250 chars):
   -------------------------------------------------------
   come ciascuno potrà vdire; quantunque siano fuori della loro vera, & natural forma. Et è cosi in fatto: 
                              percioche tutti quelli interualli, che si ritrouano in detti istrumenti, caua

---

## Summary and Next Steps

### What You've Built:
✅ A vector database of Italian music theory treatises  
✅ Semantic search capability (search by meaning, not keywords)  
✅ Intelligent incremental update system (saves time and costs)  
✅ Complete metadata tracking (author, title, date, page numbers)  

### Key Concepts:
- **TMI TEI HTML**: Thesaurus Musicarum Italicarum format for Italian scholarly texts
- **Embeddings**: Numerical representations of text meaning (3072-dimensional vectors)
- **Chunks**: Text pieces ≤2000 characters for optimal embedding
- **Vector Database**: Stores embeddings for fast similarity search
- **Semantic Search**: Finding relevant passages by meaning, not just keywords

In [ ]:
vector_store

In [ ]:
# One document in the vector store
vector_store.get(limit=1, include=['metadatas', 'documents'])